# AAOS Agent — Colab runner (vLLM + Qwen)

LLM = **vLLM serving Qwen2.5-Coder-32B (AWQ)**. Embedder = **Qwen3-Embedding-0.6B** (repo config).

**Order matters** (fixed): index runs on the **free GPU BEFORE vLLM starts** — a 0.6B embedder
on CPU is painfully slow, esp. with frameworks/base. vLLM starts only for the agent run.

0. GPU + Drive
1. Clone project + deps
2. Config (stores to Drive, agent to vLLM)
3. Choose SCOPE, clone AOSP by tag, index on GPU (skips if built)
4. Start vLLM (ensure_llm)
5. Run agent

## 0 — GPU + Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE='/content/drive/MyDrive/aaos'
STORES=f'{DRIVE}/stores'
os.makedirs(STORES,exist_ok=True)
print('STORES =',STORES)

## 1 — Clone project + deps

In [ ]:
import os
if not os.path.isdir('/content/android-auto-ai-agent'):
    !git clone https://github.com/appdev1307/android-auto-ai-agent.git /content/android-auto-ai-agent
else:
    !cd /content/android-auto-ai-agent && git pull
%cd /content/android-auto-ai-agent
!apt-get -qq install -y ripgrep >/dev/null && echo ripgrep ok
!pip -q install -r requirements.txt
print('project ready')

## 2 — Config: stores to Drive, agent to vLLM

In [ ]:
import yaml, pathlib, os
LLM_MODEL='Qwen/Qwen2.5-Coder-32B-Instruct-AWQ'
API_BASE='http://127.0.0.1:8000/v1'
cfg_path=pathlib.Path('data/config.yaml'); cfg=yaml.safe_load(cfg_path.read_text())
cfg['model']['name']=LLM_MODEL
cfg['model']['api_base']=API_BASE
cfg['rag']['stores_root']=STORES
cfg_path.write_text(yaml.safe_dump(cfg,sort_keys=False))
os.environ['OPENAI_API_KEY']='dummy'
print('LLM   =',LLM_MODEL,'@',API_BASE)
print('embed =',cfg['rag']['embed_model'])

## 3 — Choose SCOPE, clone AOSP by tag, index on GPU

`SCOPE`: `automotive` (start here, fast) | `framework` (+frameworks/base, big/slow) | `full` (bring your own tree).
Index runs on the free GPU here (vLLM not started yet) so the 0.6B embedder is fast.

In [ ]:
SCOPE = 'automotive'
TAG   = 'android-15.0.0_r1'
import os, subprocess, pathlib
AOSP='/content/aosp'
REPOS={'hardware/interfaces':'platform/hardware/interfaces',
       'packages/services/Car':'platform/packages/services/Car',
       'packages/apps/Car':'platform/packages/apps/Car',
       'frameworks/base':'platform/frameworks/base'}
SCOPE_REPOS={'automotive':['hardware/interfaces','packages/services/Car','packages/apps/Car'],
             'framework':['hardware/interfaces','packages/services/Car','packages/apps/Car','frameworks/base'],
             'full':[]}
def clone(sub,dest):
    if os.path.isdir(dest): print('  have',dest); return
    os.makedirs(pathlib.Path(dest).parent,exist_ok=True)
    subprocess.run(['git','clone','--depth=1','-b',TAG,f'https://android.googlesource.com/{sub}',dest],check=True)
for rel in SCOPE_REPOS.get(SCOPE,[]):
    clone(REPOS[rel], f'{AOSP}/{rel}')
os.environ['AOSP_ROOT']=AOSP
print('SCOPE =',SCOPE,'| TAG =',TAG,'| tree at',AOSP)
!du -sh {AOSP} 2>/dev/null

In [ ]:
import os
if os.path.exists(f'{STORES}/_base/aosp15/manifest.json'):
    print('index already on Drive -> skip (delete manifest to rebuild / change scope)')
else:
    !python -m retrieval.indexer --aosp-root {AOSP} --base --aosp-version aosp15 --scope {SCOPE}
print('index at', f'{STORES}/_base/aosp15')

## 4 — Start vLLM (ensure_llm)
Idempotent: starts vLLM only if not already on :8000. Re-run anytime it dies.

In [ ]:
import os, subprocess, time, requests
VLLM_URL='http://127.0.0.1:8000'
def _alive(p='/health'):
    try: return requests.get(VLLM_URL+p,timeout=2).ok
    except Exception: return False
def ensure_llm(max_wait=900):
    if not _alive():
        subprocess.run('pip -q install vllm', shell=True)
        cmd=('python -m vllm.entrypoints.openai.api_server '
             '--model Qwen/Qwen2.5-Coder-32B-Instruct-AWQ --quantization awq --dtype half '
             '--gpu-memory-utilization 0.90 --max-model-len 16384 --port 8000')
        subprocess.Popen(cmd+' > /content/vllm.log 2>&1', shell=True, env={**os.environ})
        for i in range(max_wait//10):
            if _alive(): break
            time.sleep(10); print(f'  vLLM starting... {(i+1)*10}s')
        else:
            print('vLLM NOT ready - log tail:')
            print(''.join(open('/content/vllm.log').readlines()[-25:]))
    print('vLLM up' if _alive() else 'vLLM DOWN')
    return VLLM_URL+'/v1'
ensure_llm()

## 5 — Run agent
Embedder on CPU here (vLLM holds GPU); query embedding is a few cheap calls. Includes the per-layer specialist pass.

In [ ]:
BUG="Android 15: VSS Vehicle.Speed not updating in HMI after ignition ON"
!CUDA_VISIBLE_DEVICES="" python -m agent.main --bug "{BUG}" --aosp-root {AOSP} --aosp-version aosp15

---
### Re-run / recovery
- New session -> cells 0,1,2, then 3 (index skips, reused), 4 (vLLM), 5.
- vLLM died -> re-run cell 4 (ensure_llm); restarts only if needed.
- Change coverage -> set SCOPE in cell 3, delete stores/_base/aosp15/manifest.json, re-run cell 3.
- First run: keep SCOPE='automotive'. framework pulls frameworks/base (big; slow first index).
- OOM on vLLM -> --max-model-len 8192 or --gpu-memory-utilization 0.85 in cell 4.
- Qwen3-Embedding load error -> pip install -U transformers sentence-transformers, re-run cell 3.